# Spark 02: Agregaciones Distribuidas y Análisis de Datos (NBA)
**Universidad del Valle de Guatemala — Data Science**

Esta libreta profundiza en el procesamiento analítico distribuido sobre datasets tabulares masivos (CSV con 84 MB de registros históricos de la NBA). Aprenderemos a manejar la inferencia de esquemas, ejecutar agregaciones globales y agrupadas (`groupBy` + `agg`), y evaluar críticamente el impacto del sesgo en el cálculo de métricas agregadas.

### 1. Verificación del Entorno de Ejecución
Comprobamos la disponibilidad de Java 17 y la versión de PySpark en el contenedor.

In [1]:
import os
import pyspark
from pyspark.sql import SparkSession


print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("PySpark:", pyspark.__version__)

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
PySpark: 3.5.1


### 2. Inicialización de la Sesión de Spark
Iniciamos el motor de Spark en modo local utilizando todos los núcleos del CPU disponibles.

In [3]:

spark = (
    SparkSession.builder
    .appName("NotebookPySpark")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

print("Spark:", spark.version)
print("Java:", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))

Spark: 3.5.1
Java: 17.0.20.1


### 3. Definición de la Carpeta de Trabajo
Fijamos la variable `mydir` apuntando al volumen montado `/opt/app/working_dir/`.

In [4]:
mydir = "/opt/app/working_dir/"

### 4. Carga Cruda de Archivo CSV (Sin Encabezado ni Esquema)
Ejecutamos una lectura básica `spark.read.csv(...)` sin opciones adicionales para diagnosticar qué ocurre cuando no se configura la lectura.
* **Observa el esquema:** Spark no sabe que la primera fila son nombres de columnas, por lo que las bautiza como `_c0, _c1, _c2...` y asigna `string` a todos los campos (incluso a los números de puntos y rebotes).

In [5]:
import pyspark.sql.functions as f

nba = spark.read.csv(mydir + 'nba/games_details.csv') 
nba.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)
 |-- _c22: string (nullable = true)
 |-- _c23: string (nullable = true)
 |-- _c24: string (nullable = true)
 |-- _c25: string (nullable = true)
 |-- _c26: string (nullable = true)
 |-- _c27: string (nullable = tru

### 5. Evidencia del Problema de la Carga Cruda
Al inspeccionar la primera fila, comprobamos que los verdaderos nombres de las columnas aparecen tratados como si fueran datos de una fila regular.

In [6]:
nba.show(1)

26/09/21 16:04:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------+-----------------+---------+---------+-----------+--------------+-------+---+---+----+------+----+----+-------+----+----+------+----+----+----+----+----+----+----+----+----+----------+
|    _c0|    _c1|              _c2|      _c3|      _c4|        _c5|           _c6|    _c7|_c8|_c9|_c10|  _c11|_c12|_c13|   _c14|_c15|_c16|  _c17|_c18|_c19|_c20|_c21|_c22|_c23|_c24|_c25|_c26|      _c27|
+-------+-------+-----------------+---------+---------+-----------+--------------+-------+---+---+----+------+----+----+-------+----+----+------+----+----+----+----+----+----+----+----+----+----------+
|GAME_ID|TEAM_ID|TEAM_ABBREVIATION|TEAM_CITY|PLAYER_ID|PLAYER_NAME|START_POSITION|COMMENT|MIN|FGM| FGA|FG_PCT|FG3M|FG3A|FG3_PCT| FTM| FTA|FT_PCT|OREB|DREB| REB| AST| STL| BLK|  TO|  PF| PTS|PLUS_MINUS|
+-------+-------+-----------------+---------+---------+-----------+--------------+-------+---+---+----+------+----+----+-------+----+----+------+----+----+----+----+----+----+----+----+----+--

### 6. Carga Correcta con `header=True` e `inferSchema=True`
Corregimos la lectura configurando los dos parámetros esenciales para datasets tabulares:
* `header=True`: Especifica que la fila inicial contiene los nombres de las columnas.
* `inferSchema=True`: Spark realiza una lectura preliminar para detectar automáticamente los tipos de datos correspondientes (enteros para minutos/puntos, decimales para porcentajes de acierto y cadenas para nombres de jugadores).

In [7]:
# Cuidado con el schema y con el encabezado
nba = spark.read.csv( 
  header = True, 
  inferSchema = True, 
  path = mydir + 'nba/games_details.csv' 
  ) 
nba.printSchema()

[Stage 3:==>                                                      (1 + 19) / 20]

root
 |-- GAME_ID: integer (nullable = true)
 |-- TEAM_ID: integer (nullable = true)
 |-- TEAM_ABBREVIATION: string (nullable = true)
 |-- TEAM_CITY: string (nullable = true)
 |-- PLAYER_ID: integer (nullable = true)
 |-- PLAYER_NAME: string (nullable = true)
 |-- START_POSITION: string (nullable = true)
 |-- COMMENT: string (nullable = true)
 |-- MIN: string (nullable = true)
 |-- FGM: double (nullable = true)
 |-- FGA: double (nullable = true)
 |-- FG_PCT: double (nullable = true)
 |-- FG3M: double (nullable = true)
 |-- FG3A: double (nullable = true)
 |-- FG3_PCT: double (nullable = true)
 |-- FTM: double (nullable = true)
 |-- FTA: double (nullable = true)
 |-- FT_PCT: double (nullable = true)
 |-- OREB: double (nullable = true)
 |-- DREB: double (nullable = true)
 |-- REB: double (nullable = true)
 |-- AST: double (nullable = true)
 |-- STL: double (nullable = true)
 |-- BLK: double (nullable = true)
 |-- TO: double (nullable = true)
 |-- PF: double (nullable = true)
 |-- PTS: dou

### 7. Verificación de Filas con Esquema Tipado
Validamos que las columnas numéricas (`PTS`, `FG_PCT`, `AST`, etc.) ya se reconocen con tipos numéricos correctos.

In [8]:
nba.show(2)

+--------+----------+-----------------+----------+---------+-------------+--------------+-------+-----+----+----+------+----+----+-------+---+---+------+----+----+---+---+---+---+---+---+----+----------+
| GAME_ID|   TEAM_ID|TEAM_ABBREVIATION| TEAM_CITY|PLAYER_ID|  PLAYER_NAME|START_POSITION|COMMENT|  MIN| FGM| FGA|FG_PCT|FG3M|FG3A|FG3_PCT|FTM|FTA|FT_PCT|OREB|DREB|REB|AST|STL|BLK| TO| PF| PTS|PLUS_MINUS|
+--------+----------+-----------------+----------+---------+-------------+--------------+-------+-----+----+----+------+----+----+-------+---+---+------+----+----+---+---+---+---+---+---+----+----------+
|42000102|1610612764|              WAS|Washington|   203078| Bradley Beal|             F|   NULL|34:36|14.0|28.0|   0.5| 1.0| 6.0|  0.167|4.0|6.0| 0.667| 0.0| 4.0|4.0|3.0|1.0|0.0|1.0|0.0|33.0|     -22.0|
|42000102|1610612764|              WAS|Washington|  1629060|Rui Hachimura|             F|   NULL|25:50| 4.0| 6.0| 0.667| 1.0| 1.0|    1.0|2.0|3.0| 0.667| 2.0| 5.0|7.0|1.0|0.0|0.0|3.0|4

---
## 🔍 Preguntas de Exploración y Agregaciones Distribuidas

### Pregunta 1: ¿Cuál es la máxima cantidad de puntos anotados por un jugador en un solo juego?

#### Paso 1.1: Agregación Global Simple (`agg(f.max(...))`)
Calculamos el valor máximo de la columna `PTS` sobre todo el dataset sin agrupar. Esto recorre todas las particiones en paralelo y computa el máximo global (81 puntos).

In [9]:
# 1. ¿Cuál es la máxima cantidad de puntos anotados por un jugador en un solo juego?
nba.agg(f.max(nba["PTS"])).show()

+--------+
|max(PTS)|
+--------+
|    81.0|
+--------+



#### Paso 1.2: Obtención de Contexto mediante Ordenamiento (`orderBy`)
Para saber quién anotó esos 81 puntos y en qué partido ocurrió, seleccionamos `game_id`, `player_name` y `pts`, y ordenamos de forma descendente.

In [10]:
nba.select("game_id","player_name","pts").orderBy(f.desc("pts")).show()

+--------+---------------+----+
| game_id|    player_name| pts|
+--------+---------------+----+
|20500591|    Kobe Bryant|81.0|
|21601076|   Devin Booker|70.0|
|20600977|    Kobe Bryant|65.0|
|20500359|    Kobe Bryant|62.0|
|22000092|  Stephen Curry|62.0|
|21300640|Carmelo Anthony|62.0|
|20300927|  Tracy McGrady|62.0|
|20800709|    Kobe Bryant|61.0|
|21901300| Damian Lillard|61.0|
|21801084|   James Harden|61.0|
|21900652| Damian Lillard|61.0|
|21300893|   LeBron James|61.0|
|21800710|   James Harden|61.0|
|21600314|  Klay Thompson|60.0|
|20600355| Gilbert Arenas|60.0|
|21501228|    Kobe Bryant|60.0|
|21900282|   James Harden|60.0|
|21800225|   Kemba Walker|60.0|
|21700748|   James Harden|60.0|
|21900125| Damian Lillard|60.0|
+--------+---------------+----+
only showing top 20 rows



#### Paso 1.3: Récord Personal de Puntos por Jugador (`groupBy` + `agg`)
Agrupamos todas las actuaciones de cada jugador mediante `groupBy("player_name")`, calculamos su puntuación máxima histórica con `agg(f.max("pts"))` y ordenamos descendentemente. Nota cómo esto involucra un proceso de **Shuffle** a través de las particiones.

In [11]:
nba.groupBy("player_name").agg(f.max("pts")).orderBy(f.desc("max(pts)")).show()

[Stage 9:========>                                                (3 + 17) / 20]

+-----------------+--------+
|      player_name|max(pts)|
+-----------------+--------+
|      Kobe Bryant|    81.0|
|     Devin Booker|    70.0|
|    Stephen Curry|    62.0|
|  Carmelo Anthony|    62.0|
|    Tracy McGrady|    62.0|
|     James Harden|    61.0|
|   Damian Lillard|    61.0|
|     LeBron James|    61.0|
|     Kemba Walker|    60.0|
|   Gilbert Arenas|    60.0|
|     Bradley Beal|    60.0|
|    Klay Thompson|    60.0|
|     Jayson Tatum|    60.0|
|    Allen Iverson|    60.0|
|    Anthony Davis|    59.0|
|Russell Westbrook|    58.0|
|     Kyrie Irving|    57.0|
|   Deron Williams|    57.0|
| Donovan Mitchell|    57.0|
|     Michael Redd|    57.0|
+-----------------+--------+
only showing top 20 rows



### Pregunta 2: En promedio, ¿cuántos puntos anota un jugador por juego?
Calculamos la media aritmética de puntos anotados con `f.avg("PTS")` para cada jugador.

💡 **Pregunta de análisis crítico para la clase:**
Observa los primeros lugares del ranking. ¿Hay jugadores poco conocidos en la cima con promedios muy altos? Esto ocurre debido al sesgo de muestras pequeñas (jugadores con 1 o 2 partidos disputados donde tuvieron buen rendimiento). En análisis de datos profesional, es indispensable aplicar un filtro de soporte (ej. `.filter(f.col("partidos_jugados") >= 50)`).

In [12]:
# 2. En promedio, ¿cuántos puntos anota un jugador por juego?
promedio_puntos_por_juego = (nba
                             .groupBy("PLAYER_NAME")
                             .agg(f.avg("PTS").alias("Promedio_Puntos"))
                             .orderBy(f.desc("Promedio_Puntos")))
promedio_puntos_por_juego.show()

+------------------+------------------+
|       PLAYER_NAME|   Promedio_Puntos|
+------------------+------------------+
|      Kevin Durant|26.827046918123276|
|      LeBron James| 26.75803517283202|
|       Kobe Bryant|26.621463414634146|
|     Allen Iverson|25.863340563991322|
|   Zion Williamson|25.619565217391305|
|       Luka Doncic| 25.51818181818182|
|      James Harden| 24.60861423220974|
|    Damian Lillard|24.293519695044473|
|       Joel Embiid|24.135313531353134|
|     Stephen Curry| 24.05095541401274|
|        Trae Young|23.690582959641254|
|     Anthony Davis| 23.66923076923077|
|  Donovan Mitchell| 23.28930817610063|
| Russell Westbrook| 22.85131459655485|
|   Carmelo Anthony|22.683937823834196|
|      Kyrie Irving|22.571005917159763|
|      Devin Booker|22.548974943052393|
|Karl-Anthony Towns|22.401360544217688|
|       Dwyane Wade|21.633564280215552|
|      Bradley Beal|21.633477633477632|
+------------------+------------------+
only showing top 20 rows



### Pregunta 3: ¿Qué precisión tienen en promedio por juego todos los jugadores de la NBA en lanzamientos de campo? (`FG_PCT`)
Calculamos la efectividad global de tiro de campo (Field Goal Percentage) promediando todas las participaciones de la historia del dataset.

In [13]:
# 3. ¿Qué precisión tienen en promedio por juego todos los jugadores de la NBA en lanzamientos en tiempo de juego? (FG_PCT)
precision_promedio = (nba.agg(f.avg("FG_PCT").alias("Precision_Promedio")))

# Muestra el resultado
precision_promedio.show()

+------------------+
|Precision_Promedio|
+------------------+
|0.4159210476805598|
+------------------+

